In [9]:
import shutup; shutup.please()
import scanpy as sc
import pandas as pd
import seaborn as sns
import anndata as ad
import matplotlib.pyplot as plt
from pybiomart import Dataset
import rpy2.robjects as ro

import sys
sys.path.append("/home/lemgui01/CellExtractor/scripts")

from rename_genes import rename_genes

percent_top = 20
n_mads = 5

In [10]:
adata = sc.read_h5ad("/home/lemgui01/CellExtractor/data/Xu_cell_j_cell_2023_11_026(bonem).h5ad")
if adata.raw is not None:
    raw_adata = adata.raw.to_adata()
    count_matrix = raw_adata[:, adata.var_names].X
else:
    count_matrix = adata.X

adata = ad.AnnData(X=count_matrix, obs=adata.obs.copy(), var=adata.var.copy())

adata.var_names = rename_genes(adata.var_names)

In [ ]:
from rpy2.robjects import pandas2ri, conversion
import rpy2.robjects as ro

df_obs = adata.obs.copy()
df_var = adata.var.copy()
df_X = pd.DataFrame(adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X,
                    index=adata.obs_names,
                    columns=adata.var_names)

with conversion.localconverter(ro.default_converter + pandas2ri.converter):
    r_df_X = ro.conversion.py2rpy(df_X)
    r_df_obs = ro.conversion.py2rpy(df_obs)
    r_df_var = ro.conversion.py2rpy(df_var)
